In [ ]:
!pip install rdkit
!pip install torch_geometric
!pip install scikit_fingerprints

In [ ]:
from sklearn.model_selection import train_test_split
from skfp.fingerprints import ECFPFingerprint
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np
from rdkit.Chem import rdFingerprintGenerator
from rdkit import DataStructs

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from rdkit import Chem

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def get_atom_features(atom):
    symbols = ['C','N','O','S','F','P','Cl','Br','I']
    v = [1 if atom.GetSymbol() == s else 0 for s in symbols]
    # atom properties
    v.append(atom.GetDegree())
    v.append(atom.GetFormalCharge())
    v.append(int(atom.GetIsAromatic()))
    # hybridization one-hot
    hyb = atom.GetHybridization()
    hyb_list = [Chem.rdchem.HybridizationType.SP,
                Chem.rdchem.HybridizationType.SP2,
                Chem.rdchem.HybridizationType.SP3,
                Chem.rdchem.HybridizationType.SP3D,
                Chem.rdchem.HybridizationType.SP3D2]
    v.extend([1 if hyb == h else 0 for h in hyb_list])
    # ring membership
    v.append(int(atom.IsInRing()))
    # implicit H count
    v.append(atom.GetTotalNumHs())
    # normalized mass
    v.append(atom.GetMass() / 200)
    return v

def smiles_to_graph(smiles, target_list):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None

    node_feats = [get_atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(node_feats, dtype=torch.float)

    edge_index = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.extend([[i, j], [j, i]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    # Y ma teraz kształt [1, 500]
    y = torch.tensor([target_list], dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)
from skfp.fingerprints import ECFPFingerprint # przykładowy z skfp

# Inicjalizujemy fingerprint z skfp
fp_gen = ECFPFingerprint(fp_size=1024, radius=2, use_pharmacophoric_invariants=True)

def smiles_to_hybrid_data(smiles, targets):
    data = smiles_to_graph(smiles, targets)
    if data is None: return None

    fp = fp_gen.transform([smiles])[0]

    # KLUCZOWE: Zmieniamy kształt z (1024,) na (1, 1024)
    data.fp = torch.tensor(fp, dtype=torch.float).view(1, -1)

    return data
def preprocessing(df):
    data_list = []
    target_cols = [c for c in df.columns if c.startswith('class_')]
    for _, row in df.iterrows():
        targets = row[target_cols].values.astype(float)
        g = smiles_to_hybrid_data(row['SMILES'], targets)
        if g:
            data_list.append(g)
    loader = DataLoader(data_list, batch_size=32, shuffle=True)

    return loader, data_list


In [ ]:
import torch
from torch.nn import Linear, BatchNorm1d
from torch_geometric.nn import GCNConv, global_mean_pool

class HybridGNN(torch.nn.Module):
    def __init__(self, hidden_channels, node_features_dim, fp_dim):
        super(HybridGNN, self).__init__()
        # GNN layers
        self.conv1 = GCNConv(node_features_dim, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, hidden_channels)

        # BatchNorm after GNN
        self.bn1 = BatchNorm1d(hidden_channels)
        self.bn2 = BatchNorm1d(hidden_channels)
        self.bn3 = BatchNorm1d(hidden_channels)

        # Combined dimension
        combined_dim = hidden_channels + fp_dim

        # Linear layers
        self.lin1 = Linear(combined_dim, 256)
        self.bn_lin1 = BatchNorm1d(256)
        self.lin2 = Linear(256, 500)  # number of classes

    def forward(self, data):
        x, edge_index, batch, fp = data.x, data.edge_index, data.batch, data.fp

        # 1. GNN forward
        x = self.conv1(x, edge_index).relu()
        x = self.bn1(x)
        x = self.conv2(x, edge_index).relu()
        x = self.bn2(x)
        x = self.conv3(x, edge_index).relu()
        x = self.bn3(x)

        x = global_mean_pool(x, batch)  # [batch_size, hidden_channels]

        # 2. FP dimension safety
        if fp.dim() == 1:
            fp = fp.view(x.size(0), -1)

        # 3. Concatenate
        combined = torch.cat([x, fp], dim=1)

        # 4. Linear layers with normalization
        out = self.lin1(combined).relu()
        out = self.bn_lin1(out)
        out = self.lin2(out)
        return out


In [ ]:
import pandas as pd
import torch
from sklearn.metrics import f1_score
# df = pd.read_parquet('chebi_submission_example.parquet')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# print(df.head())

df = pd.read_parquet('chebi_dataset_train.parquet')
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)
train_loader, train_data_list = preprocessing(df_train)
val_loader, val_data_list = preprocessing(df_val)

[21:02:52] WARNING: not removing hydrogen atom without neighbors
[21:02:52] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not removing hydrogen atom without neighbors
[21:02:54] WARNING: not r

In [ ]:
input_dim = train_data_list[0].num_node_features
model = HybridGNN(hidden_channels=64, node_features_dim=input_dim, fp_dim=1024).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss()
print("Rozpoczynanie treningu...")
model.train()
for epoch in range(1, 31):
    total_loss = 0
    for data in train_loader:
        data = data.to(DEVICE)
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs

    if epoch % 5 == 0:
        model.eval()
        val_preds = []
        val_true = []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(DEVICE)
                out = model(batch)
                val_preds.append(torch.sigmoid(out).cpu().numpy())
                val_true.append(batch.y.cpu().numpy())

        # Obliczanie Macro F1
        y_pred_bin = (np.vstack(val_preds) > 0.5).astype(int)
        y_true_bin = np.vstack(val_true).astype(int)
        score = f1_score(y_true_bin, y_pred_bin, average='macro', zero_division=0)

        print(f'Epoka: {epoch:03d} | Val Macro F1: {score:.4f}')

print("Trening zakończony!")

    # E. Zapis modelu
torch.save(model.state_dict(), 'molecule_gnn_model.pth')
print("Model zapisany jako molecule_gnn_model.pth")

Rozpoczynanie treningu...
Epoka: 005 | Val Macro F1: 0.7318
Epoka: 010 | Val Macro F1: 0.7425
Epoka: 015 | Val Macro F1: 0.7552
Epoka: 020 | Val Macro F1: 0.7450
Epoka: 025 | Val Macro F1: 0.7535
Epoka: 030 | Val Macro F1: 0.7627
Trening zakończony!
Model zapisany jako molecule_gnn_model.pth


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.loader import DataLoader
from rdkit import Chem


# Modyfikujemy funkcję, by target był opcjonalny
def smiles_to_graph_inference(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return None

    node_feats = [get_atom_features(atom) for atom in mol.GetAtoms()]
    x = torch.tensor(node_feats, dtype=torch.float)

    edge_index = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.extend([[i, j], [j, i]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

    # --- NOWA CZĘŚĆ: FINGERPRINT ---
    # Musisz mieć zainicjalizowany fp_gen (np. z skfp) w zasięgu tej funkcji
    fp = fp_gen.transform([smiles])[0]
    fp_tensor = torch.tensor(fp, dtype=torch.float).view(1, -1)

    return Data(x=x, edge_index=edge_index, fp=fp_tensor)

def main():
    MODEL_PATH = 'molecule_gnn_model.pth'
    TEST_DATA_PATH = 'chebi_dataset_test_empty.parquet'
    OUTPUT_PATH = 'predictions_3.parquet'
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # 1. Wczytanie danych (bez klas!)
    df_test = pd.read_parquet(TEST_DATA_PATH)

    print("Przetwarzanie cząsteczek...")
    test_graphs = []
    valid_indices = [] # Zapamiętujemy, które wiersze były poprawne

    for i, row in df_test.iterrows():
        g = smiles_to_graph_inference(row['SMILES'])
        if g:
            g.mol_id = row['mol_id']
            g.smiles = row['SMILES']
            test_graphs.append(g)
            valid_indices.append(i)

    loader = DataLoader(test_graphs, batch_size=64, shuffle=False)

    # 2. Ładowanie modelu
    # Upewnij się, że input_dim jest taki sam jak w treningu!
    input_dim = test_graphs[0].num_node_features
    fp_dim = test_graphs[0].fp.shape[1]
    model = HybridGNN(hidden_channels=64,node_features_dim=input_dim,
    fp_dim=fp_dim).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    # 3. Predykcja
    all_preds = []
    print("Model pracuje...")
    with torch.no_grad():
        for data in loader:
            data = data.to(DEVICE)
            logits = model(data)
            probs = torch.sigmoid(logits)
            all_preds.append(probs.cpu().numpy())

    y_probs = np.vstack(all_preds)
    # Zamieniamy na 0 lub 1
    y_pred_bin = (y_probs > 0.5).astype(int)

    # 4. Zapis wyników
    # Tworzymy nazwy kolumn class_0...class_499
    target_cols = [f'class_{i}' for i in range(500)]

    res_df = pd.DataFrame(y_pred_bin, columns=target_cols)
    res_df.insert(0, 'mol_id', [g.mol_id for g in test_graphs])
    res_df.insert(1, 'SMILES', [g.smiles for g in test_graphs])

    res_df.to_parquet(OUTPUT_PATH, index=False)
    print(f"Gotowe! Wyniki zapisano w {OUTPUT_PATH}")

if __name__ == "__main__":
    from torch_geometric.data import Data # Dodatkowy import dla skryptu
    main()
    df1 = pd.read_parquet("predictions_3.parquet")

Przetwarzanie cząsteczek...


[21:09:29] WARNING: not removing hydrogen atom without neighbors
[21:09:29] WARNING: not removing hydrogen atom without neighbors
[21:09:30] WARNING: not removing hydrogen atom without neighbors
[21:09:30] WARNING: not removing hydrogen atom without neighbors
[21:09:30] WARNING: not removing hydrogen atom without neighbors
[21:09:30] WARNING: not removing hydrogen atom without neighbors
[21:09:30] WARNING: not removing hydrogen atom without neighbors
[21:09:30] WARNING: not removing hydrogen atom without neighbors
[21:09:30] WARNING: not removing hydrogen atom without neighbors
[21:09:30] WARNING: not removing hydrogen atom without neighbors
[21:09:30] WARNING: not removing hydrogen atom without neighbors
[21:09:30] WARNING: not removing hydrogen atom without neighbors
[21:09:31] WARNING: not removing hydrogen atom without neighbors
[21:09:31] WARNING: not removing hydrogen atom without neighbors
[21:09:31] WARNING: not removing hydrogen atom without neighbors
[21:09:31] WARNING: not r

Model pracuje...
Gotowe! Wyniki zapisano w predictions_3.parquet
